# Routine to fix model error 
- Need to be run for .vol, .tem and .vdf communication files

In [13]:
import numpy as np
import struct
import os

N_SEG_IN   = 27904
N_SEG_BASE = 10999

# ── Build ever-active mask from baseline original .vol ────────────────────────
SCEN_VOL_ORIG = r'<MODEL_DIR>\com_files\com-WQexport_30min_fullperiod.vol'  # ← baseline original FLOW vol
rec_size_s    = (N_SEG_IN + 1) * 4
fsize         = os.path.getsize(SCEN_VOL_ORIG)
n_times       = fsize // rec_size_s

ever_active = np.zeros(N_SEG_IN, dtype=bool)
print('Reading baseline original .vol to find ever-active segments...')
with open(SCEN_VOL_ORIG, 'rb') as f:
    for i in range(n_times):
        f.read(4)
        raw = np.frombuffer(f.read(N_SEG_IN * 4), dtype='<f4').copy()
        ever_active |= (raw > 0)
        if (i+1) % 2000 == 0:
            print(f'  {i+1:,}/{n_times:,}...', end='\r')

all_active_idx = np.where(ever_active)[0]
print(f'\nTotal ever-active: {len(all_active_idx)}')
new_to_old = all_active_idx[:N_SEG_BASE]
print(f'Using first {len(new_to_old)} active segments')

# ── Fix per-segment files ─────────────────────────────────────────────────────
files = [
    (r'<MODEL_DIR>\com_files\com-WQexport_30min_fullperiod.vol',
     r'<MODEL_DIR>\04scen\com-WQexport_clean.vol'),
    (r'<MODEL_DIR>\com_files\com-WQexport_30min_fullperiod.tem',
     r'<MODEL_DIR>\04scen\com-WQexport_clean.tem'),
    (r'<MODEL_DIR>\com_files\com-WQexport_30min_fullperiod.vdf',
     r'<MODEL_DIR>\04scen\com-WQexport_clean.vdf'),
]

for in_path, out_path in files:
    fsize    = os.path.getsize(in_path)
    rec_size = (N_SEG_IN + 1) * 4
    n_recs   = fsize // rec_size
    print(f'\n{os.path.basename(in_path)}: {n_recs:,} timesteps')
    with open(in_path, 'rb') as f_in, \
         open(out_path, 'wb') as f_out:
        for i in range(n_recs):
            ts_bytes = f_in.read(4)
            if len(ts_bytes) < 4: break
            raw      = np.frombuffer(f_in.read(N_SEG_IN * 4),
                                     dtype='<f4').copy()
            f_out.write(ts_bytes)
            f_out.write(raw[new_to_old].astype('<f4').tobytes())
            if (i+1) % 1000 == 0 or i == n_recs-1:
                print(f'  {i+1:,}/{n_recs:,}...', end='\r')
    out_size = os.path.getsize(out_path)
    expected = n_recs * (4 + N_SEG_BASE * 4)
    print(f'\n  Size: {out_size:,} (expected {expected:,} — '
          f'{"OK" if out_size == expected else "MISMATCH"})')

# ── Verify ────────────────────────────────────────────────────────────────────
print('\nVerifying .vol...')
with open(files[0][1], 'rb') as f:
    ts  = struct.unpack('<i', f.read(4))[0]
    vol = np.frombuffer(f.read(N_SEG_BASE * 4), dtype='<f4')
print(f'Zero volume cells at t=0: {(vol == 0).sum()}')
print(f'Non-zero volume cells:    {(vol > 0).sum()}')

Reading baseline original .vol to find ever-active segments...
  14,000/15,032...
Total ever-active: 10999
Using first 10999 active segments

com-WQexport_30min_fullperiod.vol: 15,032 timesteps
  15,032/15,032...
  Size: 661,408,000 (expected 661,408,000 — OK)

com-WQexport_30min_fullperiod.tem: 15,032 timesteps
  15,032/15,032...
  Size: 661,408,000 (expected 661,408,000 — OK)

com-WQexport_30min_fullperiod.vdf: 15,031 timesteps
  15,031/15,031...
  Size: 661,364,000 (expected 661,364,000 — OK)

Verifying .vol...
Zero volume cells at t=0: 21
Non-zero volume cells:    10978


In [14]:
# Which cells are zero at t=0
zero_at_t0 = np.where(vol == 0)[0] + 1  # 1-indexed segment numbers
print(f'Zero-volume segments at t=0: {zero_at_t0}')

# Monitoring segments from .inp
monitoring = [1114, 2456, 3772, 5050, 6283, 7486, 8626, 9693, 10630,
              1635, 2974, 4283, 5543, 6766, 7964, 9078, 10121, 10973]

overlap = [s for s in monitoring if s in zero_at_t0]
print(f'Monitoring segments with zero volume at t=0: {overlap}')

Zero-volume segments at t=0: [ 97  99 108 121 153 154 255 256 272 283 288 290 303 318 319 351 357 368
 389 423 424]
Monitoring segments with zero volume at t=0: []


# Now update wind speed and solar radiation binary files length

In [6]:
import numpy as np
import struct
import os

files_to_fix = [
    (r'<MODEL_DIR>\waq-wind-noFPV.bin',
     r'<MODEL_DIR>\01scen_baseline\waq-wind-noFPV_fixed.bin'),
    (r'<MODEL_DIR>\waq-sw_radiation_flux-noFPV.bin',
     r'<MODEL_DIR>\01scen_baseline\waq-sw_radiation_flux-noFPV_fixed.bin'),
]

N_SEG_IN  = 27904
N_SEG_OUT = 10999
CHUNK     = 1000   # read this many floats at a time

for in_path, out_path in files_to_fix:
    fsize    = os.path.getsize(in_path)
    rec_size = (N_SEG_IN + 1) * 4
    n_times  = fsize // rec_size
    print(f'\n{os.path.basename(in_path)}: {n_times:,} timesteps, '
          f'{N_SEG_IN:,} → {N_SEG_OUT:,} segs')

    with open(in_path, 'rb') as f_in, \
         open(out_path, 'wb') as f_out:
        for i in range(n_times):
            # Read timestamp
            ts_bytes = f_in.read(4)
            if len(ts_bytes) < 4:
                break

            # Read segment values in chunks
            remaining = N_SEG_IN
            out_buf   = []
            segs_written = 0
            while remaining > 0:
                to_read    = min(CHUNK, remaining)
                raw_chunk  = f_in.read(to_read * 4)
                arr        = np.frombuffer(raw_chunk, dtype='<f4').copy()
                # Only keep values within N_SEG_OUT range
                if segs_written < N_SEG_OUT:
                    keep = min(to_read, N_SEG_OUT - segs_written)
                    out_buf.append(arr[:keep])
                    segs_written += keep
                remaining -= to_read

            out_vals = np.concatenate(out_buf)
            f_out.write(ts_bytes)
            f_out.write(out_vals.astype('<f4').tobytes())

            if (i+1) % 5000 == 0 or i == n_times-1:
                print(f'  {i+1:,}/{n_times:,}...', end='\r')

    out_size = os.path.getsize(out_path)
    expected = n_times * (4 + N_SEG_OUT * 4)
    print(f'\n  Size: {out_size:,} bytes '
          f'(expected {expected:,} — '
          f'{"OK" if out_size == expected else "MISMATCH"})')

print('\nDone.')


waq-wind-noFPV.bin: 79,646 timesteps, 27,904 → 10,999 segs
  79,646/79,646...
  Size: 3,504,424,000 bytes (expected 3,504,424,000 — OK)

waq-sw_radiation_flux-noFPV.bin: 554 timesteps, 27,904 → 10,999 segs
  554/554...
  Size: 24,376,000 bytes (expected 24,376,000 — OK)

Done.
